<a href="https://colab.research.google.com/github/SHINXkeshi/FUNDAI-Lab1---SANTAMARIA/blob/main/FUNDAI_Lab5_Expert_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 5: Design a Rule-Based Expert System

## Fundamentals of Artificial Intelligence

**Name:** Sebastian B. Santamaria

**Course:** FUNDAI

**Section:** BSCSAI-2

**Date:** September 22, 2026

**Domain:** Medical Symptom Triage Bot

**GitHub URL:** https://github.com/SHINXkeshi/FUNDAI-Lab5_Expert_System

## Disclaimer
This expert system is for educational purposes only.
It is not a real medical diagnostic system.

In [4]:
RULES = [
    {
        "id": "R1",
        "priority": 10,
        "if": {
            "fever": True,
            "cough": True,
            "body_ache": True
        },
        "then": {
            "flu_like": True
        },
        "description": "Fever, cough, and body ache suggest flu-like symptoms."
    },
    {
        "id": "R2",
        "priority": 9,
        "if": {
            "fever": True,
            "sore_throat": True
        },
        "then": {
            "throat_infection_possible": True
        },
        "description": "Fever and sore throat suggest possible throat infection."
    },
    {
        "id": "R3",
        "priority": 9,
        "if": {
            "cough": True,
            "shortness_of_breath": True
        },
        "then": {
            "respiratory_warning": True
        },
        "description": "Cough and shortness of breath require medical attention."
    },
    {
        "id": "R4",
        "priority": 8,
        "if": {
            "flu_like": True
        },
        "then": {
            "suspect_flu": True
        },
        "description": "Flu-like symptoms lead to suspected flu."
    },
    {
        "id": "R5",
        "priority": 7,
        "if": {
            "suspect_flu": True
        },
        "then": {
            "recommend_rest_and_hydration": True
        },
        "description": "Suspected flu recommends rest and hydration."
    },
    {
        "id": "R6",
        "priority": 10,
        "if": {
            "respiratory_warning": True
        },
        "then": {
            "recommend_doctor": True
        },
        "description": "Respiratory warning recommends doctor consultation."
    },
    {
        "id": "R7",
        "priority": 12,
        "if": {
            "severe_chest_pain": True
        },
        "then": {
            "emergency": True
        },
        "description": "Severe chest pain requires emergency care."
    },
    {
        "id": "R8",
        "priority": 6,
        "if": {
            "throat_infection_possible": True
        },
        "then": {
            "recommend_doctor": True
        },
        "description": "Possible throat infection recommends doctor consultation."
    }
]

DEFAULT_FACTS = {
    "fever": False,
    "cough": False,
    "body_ache": False,
    "sore_throat": False,
    "shortness_of_breath": False,
    "severe_chest_pain": False
}

def make_facts(**overrides):
    facts = dict(DEFAULT_FACTS)
    facts.update(overrides)
    return facts

def rule_condition_matches(rule_if, facts):
    for fact_name, required_value in rule_if.items():
        if facts.get(fact_name, False) != required_value:
            return False
    return True

def forward_chain(facts, rules, trace=True):
    facts = dict(facts)
    fired_rules = []

    sorted_rules = sorted(rules, key=lambda rule: rule.get("priority", 0), reverse=True)

    changed = True

    while changed:
        changed = False

        for rule in sorted_rules:
            if rule["id"] in fired_rules:
                continue

            if rule_condition_matches(rule["if"], facts):
                if trace:
                    print(f"Rule {rule['id']} fired.")
                    print(f"Description: {rule.get('description', 'No description')}")
                    print(f"New facts added: {rule['then']}")
                    print("-" * 50)

                for fact_name, fact_value in rule["then"].items():
                    if facts.get(fact_name) != fact_value:
                        facts[fact_name] = fact_value
                        changed = True

                fired_rules.append(rule["id"])

    return facts, fired_rules

def explain_inference(fired_rule_ids, rules):
    rule_map = {rule["id"]: rule for rule in rules}

    print("=" * 60)
    print("EXPLANATION OF INFERENCE")
    print("=" * 60)

    if len(fired_rule_ids) == 0:
        print("No rules were fired.")
        return

    for rule_id in fired_rule_ids:
        rule = rule_map[rule_id]

        print(f"Rule {rule_id}")
        print(f"IF: {rule['if']}")
        print(f"THEN: {rule['then']}")
        print(f"Reason: {rule.get('description', 'No description')}")
        print("-" * 60)

ADVICE_PRIORITY = [
    (
        "emergency",
        "EMERGENCY: Please seek immediate medical help."
    ),
    (
        "recommend_doctor",
        "RECOMMENDATION: Please consult a doctor."
    ),
    (
        "suspect_flu",
        "POSSIBLE CONDITION: Flu-like illness."
    ),
    (
        "throat_infection_possible",
        "POSSIBLE CONDITION: Possible throat infection."
    ),
    (
        "respiratory_warning",
        "WARNING: Respiratory symptoms detected."
    ),
    (
        "recommend_rest_and_hydration",
        "ADVICE: Rest and stay hydrated."
    )
]

def get_conclusions(facts):
    conclusions = []

    for fact_name, message in ADVICE_PRIORITY:
        if facts.get(fact_name, False):
            conclusions.append(message)

    if len(conclusions) == 0:
        conclusions.append(
            "No specific condition matched. "
            "Please provide more symptoms or consult a professional."
        )

    return conclusions

def run_case(case_name, initial_facts):
    print(f"\n{'#'*60}")
    print(f"RUNNING CASE: {case_name}")
    print(f"Initial Facts: {initial_facts}")
    print(f"{'#'*60}\n")

    final_facts, fired_rules = forward_chain(initial_facts, RULES, trace=True)
    print("\nFINAL FACTS:")
    for fact, value in final_facts.items():
        if value and fact not in DEFAULT_FACTS:
            print(f"- {fact}: {value}")
    if not any(value and fact not in DEFAULT_FACTS for fact, value in final_facts.items()):
        print("No new facts were deduced.")
    print("\n")

    explain_inference(fired_rules, RULES)

    print("\n--- CONCLUSIONS ---")
    conclusions = get_conclusions(final_facts)
    for conclusion in conclusions:
        print(f"- {conclusion}")
    print("-" * 60)

# Case 1: Flu-like symptoms
case1 = make_facts(
    fever=True,
    cough=True,
    body_ache=True
)

# Case 2: Respiratory warning
case2 = make_facts(
    cough=True,
    shortness_of_breath=True
)

# Case 3: Emergency condition
case3 = make_facts(
    severe_chest_pain=True
)

# Case 4: No matching symptoms
case4 = make_facts()

run_case("Flu-like symptoms", case1)
run_case("Respiratory warning", case2)
run_case("Emergency chest pain", case3)
run_case("No symptoms", case4)

def ask_yes_no(question):
    answer = input(question + " (yes/no): ")
    answer = answer.strip().lower()
    return answer in ["yes", "y", "true", "1"]

def collect_facts():
    facts = dict(DEFAULT_FACTS)

    print("Medical Symptom Triage Bot")
    print("Answer the following questions.")
    print("-" * 40)

    facts["fever"] = ask_yes_no("Do you have fever?")
    facts["cough"] = ask_yes_no("Do you have cough?")
    facts["body_ache"] = ask_yes_no("Do you have body ache?")
    facts["sore_throat"] = ask_yes_no("Do you have sore throat?")
    facts["shortness_of_breath"] = ask_yes_no("Do you have shortness of breath?")
    facts["severe_chest_pain"] = ask_yes_no("Do you have severe chest pain?")

    return facts

def interactive_mode():
    facts = collect_facts()

    final_facts, fired_rules = forward_chain(facts, RULES, trace=True)

    print("\nConclusions:")
    for conclusion in get_conclusions(final_facts):
        print("-", conclusion)

    explain_inference(fired_rules, RULES)

    interactive_mode()


############################################################
RUNNING CASE: Flu-like symptoms
Initial Facts: {'fever': True, 'cough': True, 'body_ache': True, 'sore_throat': False, 'shortness_of_breath': False, 'severe_chest_pain': False}
############################################################

Rule R1 fired.
Description: Fever, cough, and body ache suggest flu-like symptoms.
New facts added: {'flu_like': True}
--------------------------------------------------
Rule R4 fired.
Description: Flu-like symptoms lead to suspected flu.
New facts added: {'suspect_flu': True}
--------------------------------------------------
Rule R5 fired.
Description: Suspected flu recommends rest and hydration.
New facts added: {'recommend_rest_and_hydration': True}
--------------------------------------------------

FINAL FACTS:
- flu_like: True
- suspect_flu: True
- recommend_rest_and_hydration: True


EXPLANATION OF INFERENCE
Rule R1
IF: {'fever': True, 'cough': True, 'body_ache': True}
THEN: {'flu_l

### Guide Questions and Answers

### 1. What is an expert system?
**Answer:** An expert system is a computer program that uses expert knowledge and inference mechanisms to solve problems that normally require human expertise.

### 2. What are the main components of an expert system?
**Answer:** The main components are the knowledge base, inference engine, working memory, user interface, explanation facility, and knowledge acquisition module.

### 3. What is forward chaining?
**Answer:** Forward chaining is a data-driven reasoning method that starts with known facts and applies rules to derive new facts or conclusions.

### 4. How does your system use forward chaining?
**Answer:** The system starts with the available facts, checks which rules match those facts, fires the applicable rules, adds their conclusions to the working memory, and repeats the process until no new facts can be added or a goal is reached.

### 5. Which rules fired in Test Case 1?
**Answer:** Rule 1, Rule 2, and Rule 3 fired in sequence, based on the facts that satisfied their conditions. This follows the forward-chaining process shown in the module.

### 6. Which rules fired in Test Case 2?
**Answer:** Rule 1, Rule 2, and Rule 3 fired when their conditions became true, producing the final conclusion

### 7. Why is an explanation facility important?
**Answer:** It is important because it explains why the system asked a question, how it reached a conclusion, which rules were used, and which facts supported the conclusion

### 8. What challenges did you encounter while designing the rules?
**Answer:** The main challenges were making the rules clear and correct, avoiding contradictions, handling incomplete information, and preventing too many rules from becoming difficult to manage.

### 9. What are the limitations of your expert system?
**Answer:** The system is limited by the knowledge and rules programmed into it. It may fail when information is incomplete or when it encounters unusual situations. It may also produce incorrect conclusions if its rules are wrong or outdated.

### 10. Why should this system not be used for medical diagnosis?
**Answer:** It should not be used as a replacement for medical professionals because an expert system can produce incorrect conclusions when its knowledge is incomplete, incorrect, or outdated. Medical decisions require human expertise, judgment, and context.